# Classes, Inheritance & Pattern Matching — Polyglot Reference

The most divergent topic in this curriculum. Object-oriented mental models split most sharply across these six — single vs multiple inheritance, primary constructors vs declaration syntax, interfaces vs traits vs abstract classes, records vs case classes vs data classes, and a five-rung pattern-matching ladder.

**Languages:** Java · Scala · Kotlin · JavaScript · TypeScript · Python

This notebook covers:

1. **Class declaration & constructors** — class keyword, primary vs secondary constructors, field shorthand
2. **Inheritance, abstract, sealed** — `extends`, abstract classes, sealed hierarchies, modifiers
3. **Interfaces, traits, mixins** — what each language offers for shared behavior without single-base inheritance
4. **Data-shaped classes** — `record`, `case class`, `data class`, `@dataclass`, plain object literals
5. **Singletons & companions** — Scala `object`, Kotlin `companion object`, Java `static`, Python module-level
6. **Pattern matching & destructuring** — match / case / when across the languages, with destructuring on the data classes from section 4

Statement-vs-expression positioning of `match` is in `03-operators-expressions.ipynb`. The control-flow side of `match` (basic case syntax, exhaustiveness over sealed types) is in `04-control-flow.ipynb` — this notebook focuses on *destructuring* and *type patterns*. Closures and method references are in `05-functions.ipynb`.

## Class Declaration & Constructors

Where the constructor body lives, and how fields get initialized, are the loudest divergences. Java requires a separate constructor body. Scala and Kotlin have a *primary constructor* in the class header. Python uses `__init__`. JavaScript class syntax (since ES6) is sugar over prototype-chain assignment.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| class keyword | `class Foo` | `class Foo` | `class Foo` | `class Foo` | same | `class Foo:` |
| primary constructor | — *(separate method)* | in class header — `class Foo(x: Int)` | in class header — `class Foo(val x: Int)` | — *(use `constructor`)* | same | `def __init__(self, x):` |
| secondary constructor | overloaded `Foo(...)` | `def this(...)` | `constructor(...)` | — *(no overloading)* | same | — *(use defaults)* |
| field declaration | `private final int x;` | `val x` *(in body)* | `val x` *(body or header)* | `x;` *(in body)* / `this.x = ...` | `x: number` *(typed)* | `self.x = ...` *(in `__init__`)* |
| field initializer in header | — | `class Foo(val x: Int)` | `class Foo(val x: Int)` | — | parameter property — `constructor(public x: number)` | `@dataclass` *(or manual)* |
| getter / setter | manual *(or Lombok)* | implicit on `val` / `var` | implicit on `val` / `var` | `get` / `set` keywords | same | `@property` decorator |
| read-only field | `final int x` | `val x` | `val x` | `#x` *(private accessor)* | `readonly x: number` | manual *(no enforcement)* |
| mutable field | non-`final` | `var x` | `var x` | `x` | `x: number` | `self.x` *(default)* |
| static field | `static int x` | `object Foo { val x = ... }` | `companion object { val x = ... }` | `static x = ...` | same | class-body assignment |
| method | `int f() { ... }` | `def f(): Int = ...` | `fun f(): Int = ...` | `f() { ... }` | same | `def f(self): ...` |

Scala and Kotlin's *primary constructor* is the largest divergence here. The class header doubles as the constructor signature: `class Point(val x: Int, val y: Int)` declares two read-only fields and the constructor that initializes them, in one line. Java's equivalent is roughly five times longer. Java 16's `record` closes the gap for plain data classes (more on that in section 4).

TypeScript's *parameter properties* — `constructor(public x: number, private y: number)` — give the same one-liner shape using a constructor-only convention. This is the only place TypeScript's class syntax diverges from JavaScript; the runtime is plain JavaScript with explicit assignments compiled in.

## Inheritance, Abstract, Sealed

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| extends keyword | `extends` | `extends` | `:` | `extends` | same | `class Sub(Super):` |
| single vs multiple inheritance | single | single *(traits for mixins)* | single | single | single | **multiple** *(MRO via C3)* |
| call super constructor | `super(...);` *(first stmt)* | header — `class Sub(...) extends Super(...)` | header — `class Sub(...) : Super(...)` | `super(...);` | same | `super().__init__(...)` |
| call super method | `super.m()` | `super.m()` | `super.m()` | `super.m()` | same | `super().m()` |
| open for inheritance | open by default | open by default | **closed by default** *(use `open`)* | open by default | same | open by default |
| abstract class | `abstract class` | `abstract class` | `abstract class` | — *(JS has none)* | `abstract class` | `class Foo(ABC):` *(`abc` module)* |
| abstract method | `abstract T m();` | `def m(): T` *(no body)* | `abstract fun m(): T` | — | `abstract m(): T;` | `@abstractmethod` |
| sealed hierarchy | `sealed` *(17+)* with `permits` | `sealed trait` / `enum` | `sealed class` / `interface` | — | discriminated union *(simulates)* | — *(use enums or `Union`)* |
| override keyword | `@Override` *(annotation, optional)* | `override` *(required)* | `override` *(required)* | — *(implicit by name)* | `override` *(4.3+, opt-in)* | `@override` *(3.12+, hint)* |
| final / closed for override | `final` | `final` | default *(no `open`)* | — | — | — |

**Kotlin's *closed by default*** is the most opinionated choice on this table. Every class is `final` unless declared `open`; every method is `final` unless declared `open`. Inspired by Effective Java's *design and document for inheritance, or else prohibit it*. Coming from Java, this catches Kotlin newcomers constantly — you write a `class` intending to subclass it later, then can't.

**Python's multiple inheritance** is the *only* multiple inheritance here. Resolution order uses the C3 linearization algorithm — predictable but not always intuitive. Most Python codebases avoid deep multiple-inheritance hierarchies; *mixins* (small classes that add a method or two) are the common pattern.

**JavaScript has no `abstract`** at the language level. TypeScript adds `abstract class` and `abstract m()` as a compile-time check — at runtime it is a regular class. Calling `new AbstractFoo()` is a TypeScript error but a JavaScript success.

## Interfaces, Traits, Mixins

How you express *implements this contract* without single-base inheritance.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| keyword | `interface` | `trait` | `interface` | — *(folklore class-mixin pattern)* | `interface` *(structural)* | `class P(Protocol):` *(structural)* / `ABC` *(nominal)* |
| method bodies allowed | `default` methods *(8+)* | yes | yes | n/a | declarations only | yes *(via `Protocol` defaults)* |
| instance state allowed | — *(static fields only)* | yes — `val` / `var` | abstract `val` / `var` *(no backing field)* | n/a | — | yes |
| constructor parameters | — | — *(Scala 3 trait params yes)* | — | n/a | — | yes |
| multi-impl by class | yes | yes — *trait linearization* | yes | n/a | yes — declaration merging | yes — multiple Protocols / ABCs |
| structural / nominal | nominal | nominal | nominal | n/a | structural | structural *(`Protocol`)* / nominal *(`ABC`)* |
| diamond resolution | impl chooses | linearization order | explicit `super<T>.m()` | n/a | declaration-merge order | C3 MRO |
| extension function on T | — | implicit class | `fun T.f()` | — | declaration merging | — |

**Scala traits and *linearization*** are the most flexible mixin mechanism here. A `class Foo extends A with B with C` linearizes the supertype hierarchy into a single chain: when `m()` is called and only some of `A` / `B` / `C` define it, the linearization order picks which one runs. Calling `super.m()` walks the chain, not just the immediate parent. The mental model is *stackable modifications*. Powerful, but easy to get tangled.

**TypeScript interfaces are *structural*** — implementing one is automatic if your shape matches. You never *need* `implements Foo` (though you may write it). Two interfaces with identical shape are interchangeable. This is the fundamental difference from Java / Scala / Kotlin's nominal interfaces, where the name matters.

**Python `Protocol` (PEP 544, 3.8+)** added structural typing alongside the nominal `ABC` (Abstract Base Class). A `Protocol` says *anything with these methods*; an `ABC` says *anything that explicitly inherits from this class*. Both coexist; type-checking tools support both.

## Data-Shaped Classes

Each language has a sugared form for *plain data with auto-generated equality, hashing, and printing*.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| keyword | `record Foo(...)` *(16+)* | `case class Foo(...)` | `data class Foo(...)` | object literal `{x: 1}` | object literal / `interface` shape | `@dataclass class Foo:` *(3.7+)* |
| auto equality | yes *(structural)* | yes | yes | reference *(use lib for value)* | reference | yes |
| auto hashCode | yes | yes | yes | manual | manual | yes |
| auto toString | yes | yes | yes | `[object Object]` | same | yes *(`__repr__`)* |
| destructuring | record patterns *(21+)* | `Foo(a, b)` extractor | `componentN()` *(positional)* | `{x, y} = obj` *(by name)* | same | `case Foo(a, b):` *(3.10+)* |
| copy with modification | — *(write `with*` by hand)* | `foo.copy(x = 10)` | `foo.copy(x = 10)` | `{...foo, x: 10}` | same | `dataclasses.replace(foo, x=10)` |
| immutability | `final` fields *(record always)* | `val` fields *(case class always)* | `val` fields *(idiomatic)* | `Object.freeze` | `readonly` | `frozen=True` |
| inheritance | record cannot extend a class | sealed-trait + case-class hierarchy | data class cannot extend a data class | n/a | n/a | yes *(reduces auto-gen)* |
| usable as map key | yes *(equals + hashCode)* | yes | yes | reference equality bites | same | yes *(hashable when frozen)* |

**Scala `case class`** is the original on the JVM and the most powerful. Auto equality, hashing, `toString`, `copy`, an `unapply` extractor for pattern matching, JSON serialization via libraries — all from one declaration. Sealed trait plus a set of case classes is the canonical algebraic data type encoding.

**Java `record` (16+)** is the modern equivalent — but stricter. Fields are `final`, the class is `final`, no inheritance. Records cannot extend other classes (they implicitly extend `java.lang.Record`). They can implement interfaces. Java 21 added record patterns for destructuring inside `switch` and `instanceof`.

**Kotlin `data class`** sits between. Fields are typically `val` but can be `var`. Has `copy()` and `componentN()` for destructuring. Cannot inherit from another data class but can extend a regular class or implement interfaces.

**Python `@dataclass`** is a decorator that synthesizes `__init__`, `__repr__`, `__eq__`, and (with `frozen=True`) `__hash__`. More flexible than the JVM equivalents — you can add custom methods, inherit from other dataclasses, mix in defaults — and the auto-generated code is plain Python you can read in `dataclasses.py`.

**JavaScript / TypeScript** have no built-in equivalent. The plain object literal `{x: 1, y: 2}` is the canonical data shape, but equality is reference, `hashCode` does not exist, and there is no built-in copy syntax (spread `{...foo, x: 10}` is the idiom). For value-equality, use a library or stringify-and-compare.

## Singletons & Companions

How each language expresses *one instance, accessible by name* and *static-like state attached to a class*.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| singleton | enum-of-one *(idiom)* / static class | `object Foo` | `object Foo` | module-level `const` | same | module-level value |
| static method | `static` method | `object Foo { def m() = ... }` | `companion object { fun m() = ... }` | `static m()` *(in class)* | same | `@staticmethod` *(or just module fn)* |
| static field | `static` field | `object Foo { val x = ... }` | `companion object { val x = ... }` | `static x = ...` | same | class-body assignment |
| factory method | `static of(...)` | `object Foo { def apply(...) = ... }` | `companion object { operator fun invoke(...) }` | `static factory()` | same | `@classmethod` |
| class method *(receives class)* | — | — | — | — | — | `@classmethod def m(cls): ...` |
| module-level function | — *(must be `static` method)* | yes — top-level `def` | yes — top-level `fun` | yes | yes | yes |
| `this` inside companion | n/a | the singleton itself | the singleton itself | the class object | same | `cls` *(class)* / no `self` *(static)* |

**Scala `object`** is the cleanest singleton on this list. `object Foo { ... }` declares a singleton named `Foo`; you write `Foo.bar()` to call methods. There is no separate static / instance distinction — the singleton *is* the static side. A *companion object* is an `object` with the same name as a class, in the same file — it has access to private members of the class. It is where you put factory methods, constants, and the `apply` function for case-class–style construction.

**Kotlin's `companion object`** is Scala's idea adapted to JVM interop. Members of the companion are accessed as `Foo.bar()` from Kotlin code (looks like a static call) and as `Foo.Companion.bar()` from Java. The `@JvmStatic` annotation on companion members exposes them as true Java statics.

**JavaScript modules** make module-level `const` the natural singleton. Each module is loaded once; its exports are shared. Class-level `static` (since ES6) gives static methods and fields on the constructor function. The pre-ES6 IIFE-singleton pattern is now obsolete.

**Python's three method decorators.** Plain `def m(self)` is an instance method. `@staticmethod` makes it a function attached to the class — no `self`, no `cls`. `@classmethod` passes the class as the first argument — useful for alternate constructors: `@classmethod def from_string(cls, s): return cls(...)`. The first-class status of class objects in Python makes alternate-constructor patterns much more natural than the JVM languages — a subclass calling `from_string` automatically returns an instance of the subclass.

## Pattern Matching & Destructuring

The control-flow side of `match` was covered in `04-control-flow.ipynb`. This section focuses on *destructuring* — pulling fields out of a value while matching its shape — and *type patterns* — checking type and binding the narrowed value in one step.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| destructure record / case class | `case Point(x, y) ->` *(21+)* | `case Point(x, y) =>` | `componentN()` *(positional only)* | `{x, y} = obj` *(destructuring assign)* | same | `case Point(x, y):` *(3.10+)* |
| nested destructuring | `case Outer(Inner(a, b))` *(21+)* | `case Outer(Inner(a, b))` | manual | nested destructuring assign | same | `case Outer(Inner(a, b)):` |
| destructure list / sequence | — | `case List(a, b, c)` | manual | `[a, b, c] = arr` | same | `case [a, b, c]:` |
| rest pattern | — | `case List(head, tail @ _*)` | — | `[a, ...rest]` | same | `case [a, *rest]:` |
| destructure map / dict | — | extractor with `Map` | manual | `{x, y} = obj` | same | `case {"name": n}:` |
| type pattern *(test + bind)* | `case Foo f when ...` *(21+)* | `case f: Foo if ...` | `is Foo` in `when` *(smart-cast)* | manual `instanceof` | type guard | `case Foo():` |
| guard | `when expr` *(21+)* | `if expr` | inline `if` inside branch | manual | manual | `case x if x > 0:` |
| sealed exhaustiveness | sealed `switch` *(21+)* | sealed `match` | sealed `when` *(when expression)* | discriminated-union narrowing | same | structural narrowing |
| custom extractor | — | `unapply` method | — | — | — | `__match_args__` *(3.10+)* |
| capture full value | `case Foo f ->` | `case f @ Foo(...)` | `is Foo` smart-cast | manual | type-guard binding | `case f := Foo():` *(walrus)* |

**Scala has the most powerful pattern matching** of these six. Custom extractors via `unapply`, sealed-trait exhaustiveness, nested patterns, guards, and *as-patterns* (`name @ pattern` to capture both the whole and the parts) all compose. The `match` expression on a sealed trait is the canonical encoding for *visit each case and the compiler will tell me if I miss one*.

**Java 21's pattern switch** is the modern equivalent and intentionally Scala-inspired. `switch (shape) { case Circle(double r) -> Math.PI * r * r; case Square(double s) -> s * s; }` with sealed-class exhaustiveness. Records and sealed classes were added precisely to make this useful — without records there is nothing to destructure; without sealed classes there is no exhaustiveness check.

**Python's `match`** (3.10+) is *structural pattern matching*. `case Point(x=0)` matches a `Point` with `x` zero. `case [a, *rest]` destructures a sequence. `case {"name": n}` matches a dict with a `name` key. The walrus operator `case f := Foo()` captures the whole value while matching its type. The dunder attribute `__match_args__` lets a class participate as an extractor — list the field names in the order positional matching should use them.

**Kotlin's `when` is type-pattern only.** No destructuring inside `when`. Kotlin has positional destructuring elsewhere (`val (x, y) = point` via `componentN()`), but inside `when` you write `is Foo` and access fields manually after the smart-cast. Less expressive than Scala / Python / Java-21 patterns; more than enough for most code.

**JavaScript / TypeScript have no pattern matching.** TC39 has had a pattern-matching proposal in stage 1 for years. Destructuring assignment (`const {x, y} = obj`) is the closest substitute and works on objects, arrays, and nested combinations. Combine with `if` / `else if` chains and `instanceof` checks for the rest. TypeScript's discriminated unions and type guards are the closest the language gets to exhaustiveness.

## Notes — when a cell isn't enough

**Why this notebook is the largest.** OOP diverges most across the six languages. Java is the original mainstream OO design from 1995 — single inheritance, nominal interfaces, no primary constructors, no algebraic data types in the 1.0 spec. Scala layered traits and case classes on top of the JVM. Kotlin took Scala's data class, sealed types, and primary constructor and tightened them into a more opinionated shape. JavaScript's `class` is sugar over prototype chains added in ES6. TypeScript layered structural types on top. Python kept multiple inheritance and added `@dataclass` and structural `Protocol`s much later. Each language reflects when it was designed and what it inherited.

**Composition over inheritance — universal advice, Kotlin enforces it.** Every language's design guidance for the past 20 years has been *prefer composition over inheritance*. Kotlin is the only one that bakes this into syntax — classes and methods are `final` by default; you opt in to inheritance with `open`. Catches everyone the first time you write a Kotlin class meaning to subclass it later. The fix is one keyword; the cultural friction is real.

**Scala trait linearization — the diamond problem solved by ordering.** When `class Foo extends A with B with C` and multiple traits define `m()`, Scala builds a single linear chain through the supertype graph. The rightmost trait's `m()` runs first; `super.m()` walks down the chain. The order matters — `extends A with B` differs from `extends B with A`. The linearization rule is precise but takes time to internalize. Read it as *trait stacking*, where each trait wraps the next.

**Java records — what `record` is and isn't.** A `record` is a *transparent carrier of immutable data*. The compiler synthesizes the constructor, accessors named after the components (`point.x()` not `point.getX()`), `equals`, `hashCode`, and `toString`. Records are implicitly `final` and implicitly extend `java.lang.Record`. They can implement interfaces. They cannot extend other classes. They are designed to interop cleanly with switch expressions (record patterns since Java 21). Use them anywhere you would have used a Lombok `@Value` class — and, increasingly, anywhere you would have used a regular class for plain data.

**Kotlin `data class` versus Java `record`.** Both are *plain data with auto-generated boilerplate*. Differences: Kotlin's `data class` allows `var` fields (mutable); Java's `record` is always immutable. Kotlin's components are accessed as properties (`point.x`); Java's are accessed as no-arg accessor methods (`point.x()`). Kotlin's `copy()` is built in; Java's record requires hand-written `with*` methods (or Lombok). Kotlin's `componentN()` enables `val (x, y) = point` destructuring; Java's records use record patterns inside `switch` / `instanceof` instead.

**Scala `case class` versus everything else.** Adds `unapply` for use in `match` patterns. Adds an `apply` factory on the companion object — `Point(1, 2)` rather than `new Point(1, 2)`. Sealed trait + case classes is the canonical algebraic data type encoding. None of the other five capture this exact combination — Java records get close with sealed interfaces and record patterns (21+).

**Python `@dataclass` versus `attrs` versus `pydantic`.** `@dataclass` is the standard library answer (3.7+). The third-party `attrs` library predates it and has more features (validators, converters, slots optimization). `pydantic` builds on attrs / dataclasses for runtime data validation and parsing — used heavily in FastAPI, JSON-schema work, and config loading. For new code, start with `@dataclass`; reach for `attrs` if you need its features; reach for `pydantic` when you need runtime validation.

**Static vs class vs instance methods in Python.** `def m(self)` is an instance method (gets the instance). `@classmethod def m(cls)` is a class method (gets the class — useful for alternate constructors). `@staticmethod def m()` is a function attached to the class (gets nothing). The `@classmethod` form is unique among these six — Java's static methods, Scala's `object` methods, Kotlin's companion methods all receive nothing implicitly. Python's `cls` parameter makes alternate constructors clean: `Point.from_polar(r, theta)` returns a `Point`, and a subclass `ColoredPoint.from_polar(r, theta)` returns a `ColoredPoint` — without overriding the method.

**JavaScript `class` is sugar over prototypes.** Pre-ES6, you wrote `function Foo() { this.x = 1; }` and `Foo.prototype.bar = function() { ... }`. ES6 `class` syntax is the same thing with a nicer surface. There is no separate *class* concept at runtime — a class is a function whose `prototype` property holds the methods. `instanceof` walks the prototype chain. Understanding this distinction matters when you encounter older codebases or non-class objects with the same shape.

**TypeScript structural typing.** Two interfaces with identical members are interchangeable, regardless of name. A function expecting `{name: string}` accepts any object literal with a `name` string. Fundamentally different from Java / Scala / Kotlin's nominal typing where the name matters. The trade-off: structural typing reads like duck typing — works without ceremony — but unrelated types with coincidentally identical shapes silently substitute. Use *branded types* (intersection with a phantom unique tag) when you need nominal-like behavior in TypeScript.

**Pattern matching is a sliding scale (recap from topic 4).** Java 0–13: constants only. Java 14+: switch expression. Java 16+: records. Java 21+: type and record patterns plus guards. JavaScript / TypeScript: no pattern matching, just destructuring assignment. Kotlin: type and range patterns in `when`, but no nested destructuring. Python 3.10+: full structural matching including class extractors via `__match_args__`. Scala 2/3: full structural matching with custom extractors via `unapply`. Where each language sits on this ladder is worth more than memorizing the case syntax of any single one.

**Sealed hierarchies — the type-safe enum on steroids.** A sealed class or interface restricts which subclasses can extend it — declared in the same file (Kotlin), the same module (Scala), or via a `permits` clause (Java 17+). Combined with pattern matching, the compiler verifies exhaustiveness — match on a sealed type and miss a case, you get a compile warning or error. The canonical *closed sum type* encoding in OO languages. Python and JavaScript have no native equivalent — TypeScript's discriminated unions get close but rely on tag fields and assertion functions for exhaustiveness.